# Instalar dependencias

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'numpy', '--upgrade', '-q'])
subprocess.run(['pip', 'install', 'nnunetv2', 'scipy', 'nibabel', 'pandas', 'tqdm', '-q'])
print('Dependencias instaladas')


# Montar Drive y configurar rutas

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
import os
from pathlib import Path

DRIVE_BASE    = '/content/drive/MyDrive/VerSe_2020_Dataset'
DRIVE_RESULTS = f'{DRIVE_BASE}/results'
DRIVE_NNUNET  = f'{DRIVE_BASE}/MedNeXt_training/mednext'
DRIVE_PREPROC = f'{DRIVE_BASE}/preprocessed_verse_for_training'

WORKSPACE      = '/content'
NNUNET_RAW     = f'{WORKSPACE}/nnUNet_raw'
NNUNET_PREPROC = f'{WORKSPACE}/nnUNet_preprocessed'
NNUNET_RESULTS = f'{WORKSPACE}/nnUNet_results'
DATASET_NAME   = 'Dataset507_VerSe2020'
CONFIG         = '3d_lowres'

for path in [NNUNET_RAW, NNUNET_PREPROC, NNUNET_RESULTS]:
    Path(path).mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw']          = NNUNET_RAW
os.environ['nnUNet_preprocessed'] = NNUNET_PREPROC
os.environ['nnUNet_results']      = NNUNET_RESULTS
print('Rutas configuradas')


# Restaurar preprocessing desde Drive

In [ ]:
import shutil
from pathlib import Path
from tqdm.notebook import tqdm

def copy_with_shutil(src, dst, desc='Copiando'):
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    files = list(src.rglob('*'))
    for f in tqdm(files, desc=desc):
        if f.is_file():
            rel = f.relative_to(src)
            d   = dst / rel
            d.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, d)

copy_with_shutil(
    f'{DRIVE_PREPROC}/nnUNet_raw/{DATASET_NAME}',
    f'{NNUNET_RAW}/{DATASET_NAME}',
    'Copiando nnUNet_raw'
)
copy_with_shutil(
    f'{DRIVE_PREPROC}/nnUNet_preprocessed/{DATASET_NAME}',
    f'{NNUNET_PREPROC}/{DATASET_NAME}',
    'Copiando preprocessed'
)
print('Preprocessing restaurado')


# Restaurar checkpoints de MedNeXt desde Drive

In [ ]:
import shutil
from pathlib import Path

trainer_folder = f'nnUNetTrainerMedNeXt_250epochs__nnUNetPlans__{CONFIG}'
src_base = Path(DRIVE_NNUNET)   / DATASET_NAME / trainer_folder
dst_base = Path(NNUNET_RESULTS) / DATASET_NAME / trainer_folder
dst_base.mkdir(parents=True, exist_ok=True)

for fold in range(5):
    src = src_base / f'fold_{fold}'
    dst = dst_base / f'fold_{fold}'
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'fold_{fold} restaurado')
    else:
        print(f'fold_{fold} no encontrado')

print('Checkpoints restaurados')


# Instalar trainer MedNeXt 250 epochs

In [ ]:
import nnunetv2
from pathlib import Path

nnunet_dir  = Path(nnunetv2.__file__).parent
trainer_dir = nnunet_dir / 'training' / 'nnUNetTrainer' / 'variants' / 'network_architecture'
trainer_dir.mkdir(parents=True, exist_ok=True)

trainer_code = "import torch\nfrom nnunetv2.training.nnUNetTrainer.variants.network_architecture.nnUNetTrainerMedNeXt import nnUNetTrainerMedNeXt\n\nclass nnUNetTrainerMedNeXt_250epochs(nnUNetTrainerMedNeXt):\n    def __init__(self, plans, configuration, fold, dataset_json,\n                 unpack_dataset=True, device=torch.device('cuda')):\n        super().__init__(plans, configuration, fold, dataset_json,\n                         unpack_dataset, device)\n        self.num_epochs = 250\n"

trainer_path = trainer_dir / 'nnUNetTrainerMedNeXt_250epochs.py'
with open(trainer_path, 'w') as f:
    f.write(trainer_code)
print(f'Trainer instalado en {trainer_path}')


# Inferencia con ensemble de 5 folds

In [ ]:
import subprocess
from pathlib import Path

input_dir  = f'{NNUNET_RAW}/{DATASET_NAME}/imagesTs'
output_dir = f'{NNUNET_RESULTS}/{DATASET_NAME}/inference_mednext_3d_lowres'
Path(output_dir).mkdir(parents=True, exist_ok=True)

cmd = [
    'nnUNetv2_predict',
    '-d', DATASET_NAME,
    '-i', input_dir,
    '-o', output_dir,
    '-f', '0', '1', '2', '3', '4',
    '-tr', 'nnUNetTrainerMedNeXt_250epochs',
    '-c', CONFIG,
    '-p', 'nnUNetPlans',
]

print('Ejecutando inferencia MedNeXt...')
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-1000:] if result.stdout else '')
if result.returncode != 0:
    print(f'Error: {result.stderr[-500:]}')
else:
    print('Inferencia completada')


# Post Procesado

In [ ]:
import numpy as np
import nibabel as nib
from scipy import ndimage
from pathlib import Path
from tqdm.notebook import tqdm

input_dir  = Path(f'{NNUNET_RESULTS}/{DATASET_NAME}/inference_mednext_3d_lowres')
output_dir = Path(f'{NNUNET_RESULTS}/{DATASET_NAME}/inference_mednext_3d_lowres_postprocessed')
output_dir.mkdir(parents=True, exist_ok=True)

pred_files = sorted(input_dir.glob('*.nii.gz'))
print(f'Archivos a postprocesar: {len(pred_files)}')

for pred_path in tqdm(pred_files, desc='Postprocesando'):
    nii  = nib.load(str(pred_path))
    data = nii.get_fdata().astype(np.int32)
    result = np.zeros_like(data)
    for label in np.unique(data):
        if label == 0:
            continue
        binary = (data == label)
        labeled, n = ndimage.label(binary)
        if n == 0:
            continue
        sizes   = ndimage.sum(binary, labeled, range(1, n+1))
        largest = np.argmax(sizes) + 1
        result[labeled == largest] = label
    nib.save(nib.Nifti1Image(result, nii.affine, nii.header),
             str(output_dir / pred_path.name))

print(f'Postprocesado guardado en {output_dir}')


# Guardar predicciones en Drive

In [ ]:
import shutil
from pathlib import Path

src = Path(f'{NNUNET_RESULTS}/{DATASET_NAME}/inference_mednext_3d_lowres_postprocessed')
dst = Path(f'{DRIVE_RESULTS}/mednext/inference_3d_lowres_postprocessed')
dst.mkdir(parents=True, exist_ok=True)

shutil.copytree(src, dst, dirs_exist_ok=True)
n = len(list(dst.glob('*.nii.gz')))
print(f'{n} predicciones guardadas en Drive')


# Evaluacion DSC + ID Rate + RMSD

In [ ]:
import numpy as np
import nibabel as nib
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

PRED_DIR = Path(f'{DRIVE_RESULTS}/mednext/inference_3d_lowres_postprocessed')
GT_DIR   = Path(f'{DRIVE_PREPROC}/nnUNet_raw/{DATASET_NAME}/labelsTs')

pred_files = sorted(PRED_DIR.glob('*.nii.gz'))
print(f'Casos a evaluar: {len(pred_files)}')

def compute_dsc(pred, gt, label):
    p = (pred == label)
    g = (gt   == label)
    inter = (p & g).sum()
    union = p.sum() + g.sum()
    return 2 * inter / union if union > 0 else None

def compute_rmsd(pred, gt, label):
    p_vox = np.argwhere(pred == label)
    g_vox = np.argwhere(gt   == label)
    if len(p_vox) == 0 or len(g_vox) == 0:
        return None
    return float(np.sqrt(((p_vox.mean(0) - g_vox.mean(0))**2).sum()))

results = []
for pred_path in tqdm(pred_files, desc='Evaluando'):
    gt_path = GT_DIR / pred_path.name
    if not gt_path.exists():
        continue
    pred      = nib.load(str(pred_path)).get_fdata().astype(int)
    gt        = nib.load(str(gt_path)).get_fdata().astype(int)
    gt_labels = set(np.unique(gt)) - {0}
    dsc_list, rmsd_list, detected = [], [], 0
    for label in gt_labels:
        dsc = compute_dsc(pred, gt, label)
        if dsc is not None:
            dsc_list.append(dsc)
            if dsc > 0:
                detected += 1
                rmsd = compute_rmsd(pred, gt, label)
                if rmsd is not None:
                    rmsd_list.append(rmsd)
    results.append({
        'case':    pred_path.name,
        'DSC':     np.mean(dsc_list)  if dsc_list  else 0,
        'ID_Rate': detected / len(gt_labels) if gt_labels else 0,
        'RMSD_mm': np.mean(rmsd_list) if rmsd_list else 0,
    })

df = pd.DataFrame(results)
print(f'DSC:     {df["DSC"].mean():.4f}')
print(f'ID Rate: {df["ID_Rate"].mean():.4f}')
print(f'RMSD:    {df["RMSD_mm"].mean():.4f} mm')

out_csv = Path(DRIVE_RESULTS) / 'mednext' / 'evaluation_results.csv'
out_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_csv, index=False)
print(f'Guardado -> {out_csv}')
